# 教師なし学習

## アソシエーション分析（バスケット分析）

### データセットの読み込み
- 商品購買のデータを `basket_data.csv` から読み込む
  - `date`: 日付（分析には使用しない）
  - `customer_id`: 購入者
  - `item_name`: 購入商品

In [1]:
# 商品購買のデータ (cf. https://www.kaggle.com/datasets/acostasg/random-shopping-cart)
import pandas as pd
df = pd.read_csv("basket_data.csv")
display(df)

,date,customer_id,item_name
0,2000/1/1,1,yogurt
1,2000/1/1,1,pork
2,2000/1/1,1,sandwich bags
3,2000/1/1,1,lunch meat
4,2000/1/1,1,all- purpose
...,...,...,...
22338,2002/2/26,1139,soda
22339,2002/2/26,1139,laundry detergent
22340,2002/2/26,1139,vegetables
22341,2002/2/26,1139,shampoo


### データ形式の変換
- 購入者 (customer_id) ごとに，購入商品 (item_name) をまとめる

In [2]:
# customer_id 列が同じ値の行について，item_name 列の値をまとめて list にする
# (index が customer_id，値が item_name の値リストの index 付き Series になる)
dataset = df.groupby("customer_id")["item_name"].apply(list)
display(dataset)

customer_id
1       [yogurt, pork, sandwich bags, lunch meat, all-...
2       [toilet paper, shampoo, hand soap, waffles, ve...
3       [soda, pork, soap, ice cream, toilet paper, di...
4       [cereals, juice, lunch meat, soda, toilet pape...
5       [sandwich loaves, pasta, tortillas, mixes, han...
                              ...                        
1135    [sugar, beef, sandwich bags, hand soap, paper ...
1136    [coffee/tea, dinner rolls, lunch meat, spaghet...
1137    [beef, lunch meat, eggs, poultry, vegetables, ...
1138    [sandwich bags, ketchup, milk, poultry, cheese...
1139    [soda, laundry detergent, vegetables, shampoo,...
Name: item_name, Length: 1139, dtype: object

- 購入者を行，購入商品を列とする表を作り，値の True, False で誰が何を購入したかを表現する

In [3]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()

# 商品の一覧を抽出し(.fit(dataset))，2次元配列(ndarray)に変換(.transform(dataset))
#  - 各行は購入者
#  - 各列は購入商品
#  - 表の値は，購入していれば `True`，購入していなければ `False`
te_ary = te.fit(dataset).transform(dataset)

# 更に，インデックス(customer_id)と列名の付いたデータフレーム形式に変換
df2 = pd.DataFrame(te_ary, columns=te.columns_, index=dataset.index)
display(df2)

,all- purpose,aluminum foil,bagels,beef,butter,cereals,cheeses,coffee/tea,dinner rolls,dishwashing liquid/detergent,...,shampoo,soap,soda,spaghetti sauce,sugar,toilet paper,tortillas,vegetables,waffles,yogurt
customer_id,,,,,,,,,,,,,,,,,,,,,
1,True,True,False,True,True,False,False,False,True,False,...,True,True,True,False,False,False,False,True,False,True
2,False,True,False,False,False,True,True,False,False,True,...,True,False,False,False,False,True,True,True,True,True
3,False,False,True,False,False,True,True,False,True,False,...,True,True,True,True,False,True,False,True,False,False
4,True,False,False,False,False,True,False,False,False,False,...,False,False,True,False,False,True,False,False,False,False
5,True,False,False,False,False,False,False,False,True,False,...,False,False,True,True,False,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1135,True,False,False,True,False,True,True,True,True,True,...,True,True,False,False,True,False,False,False,False,False
1136,False,False,False,False,False,True,True,True,True,True,...,False,True,False,True,False,False,False,True,False,False
1137,False,False,True,True,False,False,False,False,True,True,...,True,True,False,False,True,False,True,True,False,True


### 支持度の計算
- 支持度(support)は，購入者の，顧客全体に対する比率を表す
- ここでは，購入商品の組み合わせごとに支持度(support)を計算して表にする
- すべての組み合わせは莫大な数になるので，支持度(support)が0.04(4%)以上のものに限定している

In [4]:
from mlxtend.frequent_patterns import apriori
frequet_itemsets = apriori(df2, min_support=0.04, use_colnames=True)
display(frequet_itemsets)

,support,itemsets
0,0.374890,(all- purpose)
1,0.384548,(aluminum foil)
2,0.385426,(bagels)
3,0.374890,(beef)
4,0.367867,(butter)
...,...,...
19600,0.040386,"(soda, poultry, spaghetti sauce, vegetables, l..."
19601,0.042142,"(sugar, lunch meat, toilet paper, poultry, veg..."
19602,0.042142,"(soda, lunch meat, vegetables, waffles, soap)"
19603,0.040386,"(mixes, poultry, yogurt, vegetables, milk)"


### アソシエーション分析の実行
- 前セルの支持度(support)の表をもとに，アソシエーション分析を行う
- 各行(アソシエーション・ルール)は，前提(antecedents)の商品を買った（以下事象$A$）人が結果(consequents)の商品を買う（以下，事象$C$）かどうかの情報を表す．
  各列の値は以下の通り
  - `antecedent support`: 前提の支持度．前提の商品の購入者の，顧客全体に対する比率．$P(A)$．前提の商品がどのくらい売れているか
  - `consequent support`: 結果の支持度．結果の商品の購入者の，顧客全体に対する比率．$P(C)$．結果の商品がどのくらい売れているか
  - `support`: 支持度．前提と結果の両方の商品を同時に買った人の，顧客全体に対する比率．$P(A∩C)$．前提と結果の両方の商品を同時に買った顧客がどのくらいいるか
  - `confidence`: 信頼度．$\frac{P(A∩C)}{P(A)}=P(C|A)$，前提の商品の購入者のうち，結果の商品を買った顧客はどのくらいいるか
  - `lift`: リフト値．信頼度の結果の支持度に対する比率．$\frac{P(C|A)}{P(C)}$．前提の商品を買ったという条件が加わると結果の商品を買う確率が何倍になるかということ
  - `leverage`: 影響度．支持度から前提の支持度と結果の支持度の積を引いたもの．$P(A∩C)-P(A)P(C)$．前提事象$A$と結果事象$C$の両方が起こる確率について，実際の確率と $A,C$が独立であると仮定したときの確率との差
  - `conviction`: 確信度．$\frac{1-P(C)}{1-P(C|A)}=\frac{P(\overline C)}{P(\overline C|A)}$．前提の商品を買ったという条件が加わると結果の商品を買わないという確率が何分の1になるかということ
- 前提と結果は複数の商品の組み合わせもあり，組み合わせの数が膨大になるので，リフト値(lift)が1以上の組み合わせに限定している


In [5]:
from mlxtend.frequent_patterns import association_rules

# アソシエーション分析の実行（アソシエーション・ルールの抽出）
rules = association_rules(frequet_itemsets, metric="lift", min_threshold=1)

# 支持度(support)の降順に並べ替える
rules = rules.sort_values("support", ascending=False)

# 先頭の20行のみ表示
display(rules.head(20))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
1232,(poultry),(vegetables),0.421422,0.739245,0.331870,0.787500,1.065276,0.020336,1.227083
1233,(vegetables),(poultry),0.739245,0.421422,0.331870,0.448931,1.065276,0.020336,1.049919
679,(vegetables),(eggs),0.739245,0.389816,0.326602,0.441805,1.133370,0.038433,1.093139
678,(eggs),(vegetables),0.389816,0.739245,0.326602,0.837838,1.133370,0.038433,1.607989
1362,(yogurt),(vegetables),0.384548,0.739245,0.319579,0.831050,1.124188,0.035304,1.543388
1363,(vegetables),(yogurt),0.739245,0.384548,0.319579,0.432304,1.124188,0.035304,1.084123
1360,(waffles),(vegetables),0.394205,0.739245,0.315189,0.799555,1.081583,0.023774,1.300878
1361,(vegetables),(waffles),0.739245,0.394205,0.315189,0.426366,1.081583,0.023774,1.056064
1060,(lunch meat),(vegetables),0.395083,0.739245,0.311677,0.788889,1.067155,0.019613,1.235155
1061,(vegetables),(lunch meat),0.739245,0.395083,0.311677,0.421615,1.067155,0.019613,1.045872


## クラスタリング
- 多数の参加者に，自分にとって「かわいいもの」と，それらに 17個の形容語がどのくらい当てはまるかを -2～2 の五段階で評価したデータを元に，「かわいいもの」を2つのクラスタに分けてみる (2012年に調査したデータを一部抜粋・修正)
- クラスタリングの方法は，ユークリッド距離を使った k-means 法を用いる．

### データの読み込み

In [6]:
import pandas as pd

# kawaii.csv を読み込んで表示
kawaii = pd.read_csv("kawaii.csv")
display(kawaii)

,Unnamed: 0,評定者,対象,小さい,綺麗,癒し,柔らかい,ふわふわ,あたたかい,お洒落,美しい,優しい,女の子らしい,素敵,丸い,きらきら,無邪気,和み,幼い,華やか
0,0,yO+7FddMs,妹,2,-2,2,2,2,2,-1,-1,1,1,2,2,2,2,2,2,-2
1,1,fEkFadipa,動物,-1,2,2,0,0,2,2,2,2,-1,2,0,0,2,2,0,0
2,2,Zxjd7f3yE,ねこ,0,0,2,2,2,2,0,0,0,0,0,0,0,0,2,0,0
3,3,AQyJT8wuY,女の子,1,2,2,2,2,2,2,2,2,2,-1,1,2,2,2,0,2
4,4,E2PLla6IW,ハムスター,2,-1,2,2,2,2,-1,-1,1,-1,2,2,2,2,2,2,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,251,82g9anDgk,服・小物,-1,2,0,-2,-2,-2,2,1,-2,1,2,-2,2,-2,-1,-2,2
252,252,82g9anDgk,芸能人,-2,2,-1,-2,-2,-2,2,2,0,2,2,-2,2,0,-2,-1,2
253,253,bAAu5TQlW,飼い犬,-2,1,2,2,-1,2,-2,1,1,-2,1,-1,1,2,2,-1,-1
254,254,bAAu5TQlW,赤ちゃん,2,-2,2,2,0,2,-2,-2,0,1,-2,2,2,2,2,2,-2


### クラスタリングの実行

In [7]:
from sklearn.cluster import KMeans

# 評定者と対象の行を除いたデータを用いて，クラスタリングを行い，結果を表示
# (256個の対象ごとに，クラスタ番号を付けたデータ)
kawaii_cluster = KMeans(2, random_state=0).fit(kawaii.drop(["評定者", "対象"], axis=1))
print(kawaii_cluster.labels_)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [8]:
# 元のデータ(kawaii)にクラスタの行を追加したデータを生成し，kawaii2 に代入して表示
kawaii2 = pd.concat([kawaii, pd.Series(kawaii_cluster.labels_, name="cluster")], axis=1)
display(kawaii2)

,Unnamed: 0,評定者,対象,小さい,綺麗,癒し,柔らかい,ふわふわ,あたたかい,お洒落,...,優しい,女の子らしい,素敵,丸い,きらきら,無邪気,和み,幼い,華やか,cluster
0,0,yO+7FddMs,妹,2,-2,2,2,2,2,-1,...,1,1,2,2,2,2,2,2,-2,1
1,1,fEkFadipa,動物,-1,2,2,0,0,2,2,...,2,-1,2,0,0,2,2,0,0,1
2,2,Zxjd7f3yE,ねこ,0,0,2,2,2,2,0,...,0,0,0,0,0,0,2,0,0,1
3,3,AQyJT8wuY,女の子,1,2,2,2,2,2,2,...,2,2,-1,1,2,2,2,0,2,1
4,4,E2PLla6IW,ハムスター,2,-1,2,2,2,2,-1,...,1,-1,2,2,2,2,2,2,-1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,251,82g9anDgk,服・小物,-1,2,0,-2,-2,-2,2,...,-2,1,2,-2,2,-2,-1,-2,2,0
252,252,82g9anDgk,芸能人,-2,2,-1,-2,-2,-2,2,...,0,2,2,-2,2,0,-2,-1,2,0
253,253,bAAu5TQlW,飼い犬,-2,1,2,2,-1,2,-2,...,1,-2,1,-1,1,2,2,-1,-1,0
254,254,bAAu5TQlW,赤ちゃん,2,-2,2,2,0,2,-2,...,0,1,-2,2,2,2,2,2,-2,0


### 各クラスタに属する対象を表示

In [9]:
# kawaii2 から cluster が 0 の行だけ抜粋し，対象の列のデータを表示する
print("クラスター0:", kawaii2[kawaii2["cluster"]==0]["対象"].values)
# kawaii2 から cluster が 1 の行だけ抜粋し，対象の列のデータを表示する
print("クラスター1:", kawaii2[kawaii2["cluster"]==1]["対象"].values)

クラスター0: ['うさぎ' 'その他の猫' 'ブタ' '小動物' 'ぬいぐるみ' 'ぬいぐるみ' 'サンダル' '花柄' '赤ちゃん' 'ピンク色の小物' '花'
 '赤ちゃん' '友達' 'レース' 'モデルさん' '声' '爬虫類（蛇・とかげ）' 'キャラクター' '子供' 'アクセサリー'
 '小さいアクセサリー' 'スヌーピー' 'ねこ' '犬' 'ガーリーな雑貨' '小さいもの' 'ペットのインコたちが寝てるとき'
 'スニーカーが似合うこと' '犬' 'ロリータ・ゴスロリ' '色白' '猫' '祖父母が飼っている犬' '幸せそうな笑顔' '外国の民芸品'
 '篠田麻里子' 'パンダ' '赤ちゃん' '犬' '赤ちゃん' '動物' '大型犬' '犬' '犬' '犬' 'アクセサリー' '友達'
 '外国の女の子' '海外の小物' '犬' '赤色の財布' '猫' '猫' '洋服' 'ピンク' '赤ちゃん' '吉高ゆりこ'
 '愛犬の表情、しぐさ等すべて' 'ピンク' 'ピンク色' '子供' '知念侑李' '猫' 'お花' '友達' '家で飼っている犬'
 '宮崎あおい（女優）' '北川景子' '中居正広の表情、話し方' 'ヘアアクセサリー' '猫' '動物' '少女時代' '髪型' '流行のもの'
 '赤ちゃん' 'アニメの美少女キャラ' '小物' '赤ちゃん' '赤ちゃん' 'ピアス' '女の子' '子供' '動物' 'レース'
 'ハローキティ' 'ZARA' '香水' '化粧品' 'ミッキー' 'ピンク' '芸能人' '髪色' '長谷川潤' 'ダッフィー' 'クッション'
 '服' '女の子' 'アンティーク' 'ぬいぐるみ' '犬' 'ピンク色の雑貨' 'アクセサリー' 'アクセサリー' '顔' '赤ちゃん'
 '洋服' 'アクセサリー' '小さい子ども' 'バレリーナのマスコット' '服' '化粧品' '赤ちゃん' '自分の好みに合った小物' '愛犬'
 '赤ちゃん' 'ぬいるぐみ' '安室奈美恵' 'ピンク' 'デイジー' '薔薇' '愛犬' '服・小物' '芸能人' '飼い犬' '赤ちゃん'
 '動物全般']
クラスター1: ['妹' '動物' 'ねこ' '女の子' 'ハムスター' 'おっちょこちょいな人' '子犬' '犬' '犬' '動物'